In [ ]:
get_ipython().system('pip install biopython transformers torch scikit-learn xgboost matplotlib -q')
get_ipython().system('apt-get install -y prodigal ncbi-blast+ hmmer -q')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 49.6 MB/s eta 0:00:00
Reading package lists...
Building dependency tree...
Reading state information...
The following additional packages will be installed:
  libdivsufsort3 ncbi-data
Suggested packages:
  hmmer-doc
The following NEW packages will be installed:
  hmmer libdivsufsort3 ncbi-blast+ ncbi-data prodigal
0 upgraded, 5 newly installed, 0 to remove and 53 not upgraded.
Need to get 17.6 MB of archives.
After this operation, 91.7 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libdivsufsort3 amd64 2.0.1-5 [42.8 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 ncbi-data all 6.1.20170106+dfsg1-9 [3,519 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 ncbi-blast+ amd64 2.12.0+ds-3build1 [12.3 MB]
Get:4 http://archive.ubuntu.com/ubuntu jammy/universe amd64 prodigal amd64 1:2.6.3-5 [640 kB]
Get:5 http://archive.ubuntu.com/ubuntu jammy/universe amd6

In [ ]:

import os
import re
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from Bio import SeqIO
from Bio.SeqUtils import gc_fraction
from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
import xgboost as xgb

from google.colab import drive
drive.mount('/content/drive', force_remount=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda


In [ ]:
BASE = "/content/drive/MyDrive/AI/public projects/"

pos_path      = BASE + "VF_positive_subset_final_2.faa"
neg_path      = BASE + "VF_negative_subset_final.faa"
genome_path   = BASE + "test_sample.fna"
vfdb_path     = BASE + "vfdb_proteins.faa"
pfam_hmm_path = BASE + "Pfam-A.hmm"

for p in [pos_path, neg_path, genome_path, vfdb_path, pfam_hmm_path]:
    print(f"{'OK' if os.path.exists(p) else 'MISSING'}: {p}")


OK: /content/drive/MyDrive/AI/public projects/VF_positive_subset_final_2.faa
OK: /content/drive/MyDrive/AI/public projects/VF_negative_subset_final.faa
OK: /content/drive/MyDrive/AI/public projects/test_sample.fna
MISSING: /content/drive/MyDrive/AI/public projects/vfdb_proteins.faa
MISSING: /content/drive/MyDrive/AI/public projects/Pfam-A.hmm


In [ ]:

# %% CELL 3 — Tokenizer + Dataset Class
# ------------------------------------------------------------------------------
model_name = "facebook/esm2_t12_35M_UR50D"  # upgraded from t6_8M for richer embeddings
tokenizer = AutoTokenizer.from_pretrained(model_name)

class VFTransformerDataset(Dataset):
    def __init__(self, sequences, labels, tokenizer, max_length=512):
        self.labels = labels
        self.encodings = tokenizer(
            sequences, truncation=True, padding='max_length',
            max_length=max_length, return_tensors="pt"
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.float32)
        return item



config.json:   0%|          | 0.00/778 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/95.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/93.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

In [ ]:


# %% CELL 4 — Load VF Positive/Negative Sequences + Split
# ------------------------------------------------------------------------------
sequences, labels = [], []

if os.path.exists(pos_path):
    for record in SeqIO.parse(pos_path, "fasta"):
        sequences.append(str(record.seq))
        labels.append(1)

if os.path.exists(neg_path):
    for record in SeqIO.parse(neg_path, "fasta"):
        sequences.append(str(record.seq))
        labels.append(0)

print(f"Total sequences: {len(sequences)}  |  Positive: {sum(labels)}  Negative: {len(labels)-sum(labels)}")

train_seqs, val_seqs, train_labels, val_labels = train_test_split(
    sequences, labels, test_size=0.2, random_state=42, stratify=labels
)

train_dataset = VFTransformerDataset(train_seqs, train_labels, tokenizer, max_length=512)
val_dataset   = VFTransformerDataset(val_seqs, val_labels, tokenizer, max_length=512)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=16, shuffle=False)



Total sequences: 22207  |  Positive: 11584  Negative: 10623


In [ ]:

# %% CELL 5 — Model Architecture
# ------------------------------------------------------------------------------
class AttentionPooling(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attention = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, x, mask):
        scores = self.attention(x).squeeze(-1)
        scores = scores.masked_fill(mask == 0, -1e9)
        weights = F.softmax(scores, dim=-1)
        context_vector = torch.sum(x * weights.unsqueeze(-1), dim=1)
        return context_vector, weights


class GeneEncoder(nn.Module):
    """Per-protein encoder: ESM-2 (partially unfrozen) + multi-layer weighted
    combination + attention pooling -> single embedding per protein."""
    def __init__(self, transformer_model_name, unfreeze_last_n_layers=2):
        super().__init__()
        self.transformer = AutoModel.from_pretrained(transformer_model_name)
        self.hidden_dim = self.transformer.config.hidden_size
        self.n_layers = self.transformer.config.num_hidden_layers

        for param in self.transformer.parameters():
            param.requires_grad = False

        for name, param in self.transformer.named_parameters():
            for i in range(self.n_layers - unfreeze_last_n_layers, self.n_layers):
                if f"layer.{i}." in name:
                    param.requires_grad = True

        self.layer_weight_logits = nn.Parameter(torch.zeros(self.n_layers + 1))
        self.attn_pool = AttentionPooling(self.hidden_dim)

    def forward(self, input_ids, attention_mask):
        outputs = self.transformer(
            input_ids=input_ids, attention_mask=attention_mask,
            output_hidden_states=True
        )
        all_layers = torch.stack(outputs.hidden_states, dim=0)
        layer_weights = F.softmax(self.layer_weight_logits, dim=0)
        combined = (all_layers * layer_weights.view(-1, 1, 1, 1)).sum(0)

        context_vector, attn_weights = self.attn_pool(combined, attention_mask)
        return context_vector, attn_weights, layer_weights


class VFClassifierHead(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(hidden_dim, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.net(x)


class GenomicWindowEncoder(nn.Module):
    """Takes per-gene embeddings within a genomic window, learns which
    neighboring genes matter most (e.g. secretion machinery near a toxin gene)."""
    def __init__(self, gene_embed_dim, n_heads=4, n_layers=1):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=gene_embed_dim, nhead=n_heads,
            dim_feedforward=gene_embed_dim * 2, dropout=0.1, batch_first=True
        )
        self.context_transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.window_pool = AttentionPooling(gene_embed_dim)
        self.window_classifier = nn.Sequential(
            nn.Linear(gene_embed_dim, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

    def forward(self, gene_embeddings, gene_mask):
        src_key_padding_mask = (gene_mask == 0)
        contextualized = self.context_transformer(
            gene_embeddings, src_key_padding_mask=src_key_padding_mask
        )
        window_embedding, gene_importance = self.window_pool(contextualized, gene_mask)
        window_logits = self.window_classifier(window_embedding)
        return window_embedding, gene_importance, window_logits


class FullVFPipeline(nn.Module):
    def __init__(self, transformer_model_name, unfreeze_last_n_layers=2):
        super().__init__()
        self.gene_encoder = GeneEncoder(transformer_model_name, unfreeze_last_n_layers)
        self.classifier = VFClassifierHead(self.gene_encoder.hidden_dim)
        self.window_encoder = GenomicWindowEncoder(self.gene_encoder.hidden_dim)

    def encode_gene(self, input_ids, attention_mask):
        context_vector, attn_weights, layer_weights = self.gene_encoder(input_ids, attention_mask)
        logits = self.classifier(context_vector)
        return logits, context_vector, attn_weights

    def encode_window(self, gene_embeddings, gene_mask):
        return self.window_encoder(gene_embeddings, gene_mask)




In [ ]:

# %% CELL 6 — Contrastive Loss
# ------------------------------------------------------------------------------
def contrastive_loss(embeddings, labels, temperature=0.1):
    embeddings = F.normalize(embeddings, dim=1)
    labels = labels.view(-1)

    sim_matrix = torch.matmul(embeddings, embeddings.T) / temperature
    pos_mask = (labels.unsqueeze(0) == labels.unsqueeze(1)).float()
    pos_mask.fill_diagonal_(0)

    sim_matrix = sim_matrix - sim_matrix.max(dim=1, keepdim=True)[0].detach()
    exp_sim = torch.exp(sim_matrix)
    exp_sim = exp_sim * (1 - torch.eye(exp_sim.size(0), device=exp_sim.device))

    log_prob = sim_matrix - torch.log(exp_sim.sum(dim=1, keepdim=True) + 1e-8)
    loss = -(pos_mask * log_prob).sum(1) / (pos_mask.sum(1).clamp(min=1))
    return loss.mean()




In [ ]:
print("CUDA available:", torch.cuda.is_available())
print("Device:", device)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

CUDA available: True
Device: cuda
Tesla T4


In [ ]:
print("Train batches:", len(train_loader), " Val batches:", len(val_loader))
print("Total train sequences:", len(train_seqs))

Train batches: 1111  Val batches: 278
Total train sequences: 17765


In [ ]:
from IPython.display import Javascript
Javascript('''
function ClickConnect(){
  console.log("Keeping session alive");
  document.querySelector("colab-toolbar-button#connect").click()
}
setInterval(ClickConnect, 60000)
''')

<IPython.core.display.Javascript object>

In [ ]:
from tqdm.notebook import tqdm

import os

CHECKPOINT_DIR = "/content/drive/MyDrive/AI/public projects/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
checkpoint_path = f"{CHECKPOINT_DIR}/vf_pipeline_checkpoint.pth"

model = FullVFPipeline(model_name, unfreeze_last_n_layers=2).to(device)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=2e-5)

n_pos = sum(train_labels)
n_neg = len(train_labels) - n_pos
pos_weight = torch.tensor([n_neg / n_pos]).to(device)
bce_criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

contrastive_weight = 0.3
epochs = 6  # reduced — AUC was already flattening by epoch 4
best_val_auc = 0.0
start_epoch = 0

# --- Resume logic: pick up where you left off if a checkpoint exists ---
if os.path.exists(checkpoint_path):
    ckpt = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    start_epoch = ckpt["epoch"] + 1
    best_val_auc = ckpt["best_val_auc"]
    print(f"Resumed from checkpoint: epoch {start_epoch}, best_val_auc={best_val_auc:.4f}")
else:
    print("No checkpoint found — starting fresh.")

print("\n--- Stage 1: Gene-Level Training ---")
for epoch in range(start_epoch, epochs):
    model.train()
    train_loss = 0.0
    all_train_preds, all_train_labels = [], []

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1} training"):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels_b = batch['labels'].to(device).unsqueeze(1)

        optimizer.zero_grad()
        logits, embeddings, _ = model.encode_gene(input_ids, attention_mask)
        loss = bce_criterion(logits, labels_b) + contrastive_weight * contrastive_loss(embeddings, labels_b)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        preds = (probs >= 0.5).astype(int)
        all_train_preds.extend(preds)
        all_train_labels.extend(labels_b.cpu().numpy())

    train_acc = accuracy_score(all_train_labels, all_train_preds)

    model.eval()
    val_loss = 0.0
    all_val_probs, all_val_preds, all_val_labels = [], [], []

    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels_b = batch['labels'].to(device).unsqueeze(1)

            logits, embeddings, _ = model.encode_gene(input_ids, attention_mask)
            loss = bce_criterion(logits, labels_b) + contrastive_weight * contrastive_loss(embeddings, labels_b)
            val_loss += loss.item()

            probs = torch.sigmoid(logits).cpu().numpy()
            preds = (probs >= 0.5).astype(int)
            all_val_probs.extend(probs)
            all_val_preds.extend(preds)
            all_val_labels.extend(labels_b.cpu().numpy())

    val_acc = accuracy_score(all_val_labels, all_val_preds)
    val_f1 = f1_score(all_val_labels, all_val_preds)
    val_auc = roc_auc_score(all_val_labels, all_val_probs)

    if val_auc > best_val_auc:
        best_val_auc = val_auc

    # Save EVERY epoch to Drive — survives disconnects
    torch.save({
        "epoch": epoch,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "best_val_auc": best_val_auc,
        "val_auc": val_auc,
    }, checkpoint_path)

    print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {train_loss/len(train_loader):.4f} Acc: {train_acc:.3f} | "
          f"Val Loss: {val_loss/len(val_loader):.4f} Acc: {val_acc:.3f} F1: {val_f1:.3f} AUC: {val_auc:.3f}")
    print(f"Checkpoint saved to Drive (epoch {epoch+1})")

print(f"\nTraining complete. Best Validation AUC: {best_val_auc:.4f}")

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: facebook/esm2_t12_35M_UR50D
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


No checkpoint found — starting fresh.

--- Stage 1: Gene-Level Training ---


Epoch 1 training:   0%|          | 0/1111 [00:00<?, ?it/s]

Epoch [1/6] | Train Loss: 1.3145 Acc: 0.727 | Val Loss: 1.1398 Acc: 0.831 F1: 0.836 AUC: 0.909
Checkpoint saved to Drive (epoch 1)


...

Epoch [4/6] | Train Loss: 0.9703 Acc: 0.907 | Val Loss: 1.0135 Acc: 0.894 F1: 0.900 AUC: 0.951
Checkpoint saved to Drive (epoch 4)


Epoch 5 training:   0%|          | 0/1111 [00:00<?, ?it/s]

Epoch [5/6] | Train Loss: 0.9342 Acc: 0.920 | Val Loss: 1.0113 Acc: 0.893 F1: 0.901 AUC: 0.954
Checkpoint saved to Drive (epoch 5)


Epoch 6 training:   0%|          | 0/1111 [00:00<?, ?it/s]

Epoch [6/6] | Train Loss: 0.9090 Acc: 0.928 | Val Loss: 0.9894 Acc: 0.901 F1: 0.906 AUC: 0.957
Checkpoint saved to Drive (epoch 6)

Training complete. Best Validation AUC: 0.9569


In [ ]:

# %% CELL 8 — Predict ORFs from the Genome (Prodigal)
# ------------------------------------------------------------------------------
get_ipython().system('prodigal -i "{genome_path}" -a /content/predicted_orfs.faa -f gff -o /content/predicted_orfs.gff -p meta')
print("Prodigal finished.")



Streaming output truncated to the last 5000 lines.
Finding genes in sequence #1317 (1129 bp)...done!
Finding genes in sequence #1318 (1743 bp)...done!
Finding genes in sequence #1319 (525 bp)...done!
Finding genes in sequence #1320 (705 bp)...done!
Finding genes in sequence #1321 (527 bp)...done!
Finding genes in sequence #1322 (752 bp)...done!
Finding genes in sequence #1323 (305 bp)...done!
Finding genes in sequence #1324 (325 bp)...done!
Finding genes in sequence #1325 (514 bp)...done!
Finding genes in sequence #1326 (1726 bp)...done!
Finding genes in sequence #1327 (1743 bp)...done!
Finding genes in sequence #1328 (428 bp)...done!
Finding genes in sequence #1329 (591 bp)...done!
Finding genes in sequence #1330 (1819 bp)...done!
Finding genes in sequence #1331 (3130 bp)...done!
Finding genes in sequence #1332 (1707 bp)...done!
Finding genes in sequence #1333 (460 bp)...done!
Finding genes in sequence #1334 (415 bp)...done!
Finding genes in sequence #1335 (1094 bp)...done!
Finding ge

In [ ]:


# %% CELL 9 — Parse ORF Coordinates + Load Sequences + ID-Matching Diagnostic
# ------------------------------------------------------------------------------
def parse_prodigal_gff(gff_path):
    records = []
    with open(gff_path) as f:
        for line in f:
            if line.startswith("#") or not line.strip():
                continue
            fields = line.strip().split("\t")
            contig, source, feature, start, end, score, strand, frame, attrs = fields
            id_match = re.search(r'ID=([^;]+)', attrs)
            gene_id = f"{contig}_{id_match.group(1)}" if id_match else None
            records.append({
                "gene_id": gene_id, "contig": contig,
                "start": int(start), "end": int(end), "strand": strand
            })
    return pd.DataFrame(records)

orf_coords = parse_prodigal_gff("/content/predicted_orfs.gff")
orf_coords = orf_coords.sort_values(["contig", "start"]).reset_index(drop=True)
print(f"Total ORFs: {len(orf_coords)}")

orf_seqs = {}
for record in SeqIO.parse("/content/predicted_orfs.faa", "fasta"):
    orf_seqs[record.id] = str(record.seq).rstrip("*")

print(f"Loaded {len(orf_seqs)} protein sequences")

# Diagnostic: verify GFF and FASTA gene_ids actually match
gff_ids = set(orf_coords['gene_id'].tolist())
faa_ids = set(orf_seqs.keys())
overlap = gff_ids.intersection(faa_ids)

print(f"GFF IDs: {len(gff_ids)}  |  FAA IDs: {len(faa_ids)}  |  Overlap: {len(overlap)}")

if len(overlap) == 0:
    print("WARNING: IDs DO NOT MATCH. Falling back to positional mapping.")
    faa_records = list(SeqIO.parse("/content/predicted_orfs.faa", "fasta"))
    if len(faa_records) == len(orf_coords):
        orf_seqs = {
            gid: str(rec.seq).rstrip("*")
            for gid, rec in zip(orf_coords['gene_id'].tolist(), faa_records)
        }
        print(f"Remapped by position. New overlap check: "
              f"{len(set(orf_coords['gene_id']).intersection(orf_seqs.keys()))}")
    else:
        raise ValueError(
            f"Record count mismatch: GFF has {len(orf_coords)} entries, "
            f"FAA has {len(faa_records)}. Cannot safely realign — inspect files manually."
        )
elif len(overlap) < min(len(gff_ids), len(faa_ids)):
    print("WARNING: Partial overlap — some genes will be dropped downstream.")
    print("GFF-only sample:", list(gff_ids - faa_ids)[:3])
    print("FAA-only sample:", list(faa_ids - gff_ids)[:3])
else:
    print("IDs match cleanly.")



Total ORFs: 9861
Loaded 9861 protein sequences
GFF IDs: 9861  |  FAA IDs: 9861  |  Overlap: 0
Remapped by position. New overlap check: 9861


In [ ]:

# %% CELL 10 — Build 10kb Genomic Windows
# ------------------------------------------------------------------------------
WINDOW_SIZE = 10_000

def build_genomic_windows(orf_coords, window_size=WINDOW_SIZE):
    windows = []
    for contig, group in orf_coords.groupby("contig"):
        group = group.sort_values("start")
        max_pos = group["end"].max()
        n_windows = int(max_pos // window_size) + 1

        for w in range(n_windows):
            w_start = w * window_size
            w_end = w_start + window_size
            genes_in_window = group[(group["start"] >= w_start) & (group["start"] < w_end)]
            if len(genes_in_window) >= 2:
                windows.append({
                    "contig": contig, "window_start": w_start, "window_end": w_end,
                    "gene_ids": genes_in_window["gene_id"].tolist()
                })
    return windows

genomic_windows = build_genomic_windows(orf_coords)
print(f"Built {len(genomic_windows)} genomic windows")
if genomic_windows:
    avg_genes = sum(len(w['gene_ids']) for w in genomic_windows) / len(genomic_windows)
    print(f"Avg genes per window: {avg_genes:.1f}")




Built 1908 genomic windows
Avg genes per window: 2.9


In [ ]:

# %% CELL 11 — Embed Every Predicted ORF
# ------------------------------------------------------------------------------
def get_all_orf_embeddings(orf_seqs, gene_encoder, tokenizer, device, batch_size=16, max_length=512):
    gene_ids = list(orf_seqs.keys())
    sequences = [orf_seqs[gid] for gid in gene_ids]

    embeddings = {}
    gene_encoder.eval()
    with torch.no_grad():
        for i in range(0, len(sequences), batch_size):
            batch_seqs = sequences[i:i+batch_size]
            batch_ids = gene_ids[i:i+batch_size]

            enc = tokenizer(batch_seqs, truncation=True, padding='max_length',
                             max_length=max_length, return_tensors="pt")
            input_ids = enc["input_ids"].to(device)
            attention_mask = enc["attention_mask"].to(device)

            context_vector, _, _ = gene_encoder(input_ids, attention_mask)
            for gid, emb in zip(batch_ids, context_vector.cpu()):
                embeddings[gid] = emb

    return embeddings

orf_embeddings = get_all_orf_embeddings(orf_seqs, model.gene_encoder, tokenizer, device)
print(f"Embedded {len(orf_embeddings)} ORFs")




Embedded 9861 ORFs


In [ ]:
BASE = "/content/drive/MyDrive/AI/public projects/"

# Check BLAST results
with open(BASE + "orfs_vs_vfdb_full.tsv") as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i >= 3:
            break

print("\n---\n")

# Check Pfam/HMM results
with open(BASE + "pfam_hits.tbl") as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i >= 5:
            break

ORF_41	VFG002521(gb|WP_004525693)	27.128	188	121	4	101	278	348	529	8.62e-11	61.2	40
ORF_41	VFG038687(gb|WP_016350057)	29.557	203	125	8	100	295	256	447	1.60e-10	60.1	44
ORF_41	VFG007586(gb|WP_001881782)	28.729	181	119	5	101	278	284	457	2.28e-09	56.6	40
ORF_41	VFG001264(gb|NP_250144)	25.243	206	130	6	101	296	211	402	3.35e-09	55.8	44

---

#                                                                            --- full sequence --- -------------- this domain -------------   hmm coord   ali coord   env coord
# target name        accession   tlen query name           accession   qlen   E-value  score  bias   #  of  c-Evalue  i-Evalue  score  bias  from    to  from    to  from    to  acc description of target
#------------------- ---------- ----- -------------------- ---------- ----- --------- ------ ----- --- --- --------- --------- ------ ----- ----- ----- ----- ----- ----- ----- ---- ---------------------
PcfK                 PF14058.9    125 ORF_1                -            141   3

In [ ]:
import pandas as pd

blast_path = BASE + "orfs_vs_vfdb_full.tsv"

# Standard BLAST outfmt 6 columns — adjust if your inspection above shows a different order
blast_cols = ["query_id", "subject_id", "pident", "length", "mismatch",
              "gapopen", "qstart", "qend", "sstart", "send", "evalue", "bitscore"]

blast_hits = pd.read_csv(blast_path, sep="\t", names=blast_cols, header=None)

# If the file already had a header row, use this instead:
# blast_hits = pd.read_csv(blast_path, sep="\t")

print(f"Total BLAST hits: {len(blast_hits)}")
print(f"Unique ORFs with a VFDB hit: {blast_hits['query_id'].nunique()}")
print(blast_hits.head())

best_hits = blast_hits.sort_values("evalue").drop_duplicates("query_id", keep="first")

EVALUE_THRESHOLD = 1e-10
PIDENT_THRESHOLD = 40

confident_vf_genes = set(
    best_hits[
        (best_hits["evalue"] <= EVALUE_THRESHOLD) &
        (best_hits["pident"] >= PIDENT_THRESHOLD)
    ]["query_id"]
)
print(f"Genes with confident VFDB homology: {len(confident_vf_genes)}")

Total BLAST hits: 11516
Unique ORFs with a VFDB hit: 679
                          query_id  subject_id  pident  length  mismatch  \
ORF_41  VFG002521(gb|WP_004525693)      27.128     188     121         4   
ORF_41  VFG038687(gb|WP_016350057)      29.557     203     125         8   
ORF_41  VFG007586(gb|WP_001881782)      28.729     181     119         5   
ORF_41     VFG001264(gb|NP_250144)      25.243     206     130         6   
ORF_41  VFG001895(gb|YP_002343528)      25.472     212     140         8   

        gapopen  qstart  qend  sstart          send  evalue  bitscore  
ORF_41      101     278   348     529  8.620000e-11    61.2        40  
ORF_41      100     295   256     447  1.600000e-10    60.1        44  
ORF_41      101     278   284     457  2.280000e-09    56.6        40  
ORF_41      101     296   211     402  3.350000e-09    55.8        44  
ORF_41       92     294   272     474  4.290000e-09    55.5        45  
Genes with confident VFDB homology: 0


In [ ]:
gff_gene_ids = set(orf_coords['gene_id'].tolist())
blast_query_ids = set(blast_hits['query_id'].tolist())
overlap = gff_gene_ids.intersection(blast_query_ids)
print(f"GFF genes: {len(gff_gene_ids)} | BLAST query IDs: {len(blast_query_ids)} | Overlap: {len(overlap)}")

GFF genes: 9861 | BLAST query IDs: 679 | Overlap: 0


In [ ]:
def parse_hmmer_tblout_file(tblout_path, evalue_thresh=1e-5):
    """Parses a --tblout style HMMer output file already saved to disk."""
    domain_counts = {}
    domain_hits = {}
    with open(tblout_path) as f:
        for line in f:
            if line.startswith("#") or not line.strip():
                continue
            parts = line.split()
            if len(parts) < 13:
                continue
            pfam_acc = parts[1].split(".")[0]
            gene_id = parts[2]
            domain_evalue = float(parts[4])

            if domain_evalue <= evalue_thresh:
                domain_counts[gene_id] = domain_counts.get(gene_id, 0) + 1
                domain_hits.setdefault(gene_id, set()).add(pfam_acc)
    return domain_counts, domain_hits

genome_pfam_counts, genome_pfam_hits = parse_hmmer_tblout_file(BASE + "pfam_hits.tbl")
print(f"Genome ORFs with >=1 Pfam domain: {len(genome_pfam_counts)}")

# pfam_vf_hits.tbl looks like it may already be VF-specific domain hits — worth using directly
vf_pfam_counts, vf_pfam_hits = parse_hmmer_tblout_file(BASE + "pfam_vf_hits.tbl")
print(f"Genome ORFs with VF-associated Pfam domain: {len(vf_pfam_counts)}")

ValueError: could not convert string to float: '-'